# Step 4 check: the examiner's opinion on every family of the sector

**Why.** For the future score of Step 4 we want a sign of novelty that exists for recent patents, which have no citations yet. One is the opinion of the examiner. In the search report the examiner lists the earlier documents closest to the invention and gives each a letter: **X** means the document alone already questions the novelty or the inventive step of the claims, **Y** that it does so combined with another document (the EPO also uses **I**, like X), **A** that it is only background. A family whose search report has an X or Y was close to something that already existed; a family without them was more novel in the examiner's opinion.

**Why not EPAB.** Notebook 20 (Section 8) read the search reports in the full text of EPAB, but EPAB holds only EP patents, with the reports as text only from mid-2012: only 76 of our 2,907 families are filed at the EPO, and only 11 have a report there. Far too few.

**What we do instead.** PATSTAT records the letters of the search reports of every office that sends them, in table TLS215 (one row per cited document of a publication, with its category). Step 3 already uses it in the other direction, for the blocking patents (how often our families are cited as X against later patents). Here we go backwards: for every publication of the 2,907 families, any office, we take the letters of its own search report, and for each family we record whether a report exists and whether it holds an X, Y or I. We count how many families are covered, from which offices, and year by year, to know for which years the measure can be used.

In [ ]:
# Step 4 check: the examiner's opinion on every family of the sector, from
# the search reports of all offices in PATSTAT (TLS215), not only EPAB
from pathlib import Path

import pandas as pd
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (TLS211_PAT_PUBLN,
                                                 TLS215_CITN_CATEG)

DATA_DIR = Path("data/PROD/humanoid_robots_v2")          # the tables of Step 0


def load(name):
    for suffix in (".parquet", ".pkl"):
        path = DATA_DIR / f"{name}{suffix}"
        if path.exists():
            if suffix == ".parquet":
                return pd.read_parquet(path)
            return pd.read_pickle(path)
    raise FileNotFoundError(f"{name} not in {DATA_DIR}")


patstat = PatstatClient(env="PROD")
db = patstat.orm()
families = load("families")
applications = load("applications")


def in_chunks(ids, make_query, chunk=10_000):
    ids = [int(i) for i in pd.unique(pd.Series(ids))]
    parts = [patstat.df(make_query(ids[s:s + chunk]))
             for s in range(0, len(ids), chunk)]
    parts = [p for p in parts if len(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


pubs = in_chunks(applications["appln_id"], lambda part: db.query(
    TLS211_PAT_PUBLN.pat_publn_id, TLS211_PAT_PUBLN.appln_id,
    TLS211_PAT_PUBLN.publn_auth).filter(TLS211_PAT_PUBLN.appln_id.in_(part)))
pubs["docdb_family_id"] = pubs["appln_id"].map(
    applications.set_index("appln_id")["docdb_family_id"])

cats = in_chunks(pubs["pat_publn_id"], lambda part: db.query(
    TLS215_CITN_CATEG.pat_publn_id, TLS215_CITN_CATEG.citn_categ)
    .filter(TLS215_CITN_CATEG.pat_publn_id.in_(part)))
cats = cats.merge(pubs[["pat_publn_id", "publn_auth", "docdb_family_id"]],
                  on="pat_publn_id")
cats["x_or_y"] = cats["citn_categ"].astype(str).str.contains("[XYI]")

fam = cats.groupby("docdb_family_id").agg(x_or_y=("x_or_y", "max"))
year = families.set_index("docdb_family_id")["priority_year"]
fam["priority_year"] = fam.index.map(year)
print(f"families with a search report with categories: {len(fam):,} "
      f"of {len(families):,} ({len(fam) / len(families):.0%}); with EPAB: 11")
print("office of the reports (families):",
      cats.groupby("publn_auth")["docdb_family_id"].nunique()
      .sort_values(ascending=False).head(6).to_dict())
by_year = pd.DataFrame({
    "families": families.groupby("priority_year").size(),
    "with a report": fam.groupby("priority_year").size(),
    "with an X or Y": fam[fam["x_or_y"]].groupby("priority_year").size(),
}).fillna(0).astype(int)
by_year["coverage (%)"] = (100 * by_year["with a report"]
                           / by_year["families"]).round(1)
by_year.loc[2005:]

**What the PROD run shows.** The letters of PATSTAT cover **1,700 of the 2,907 families (58%)**, against the 11 of EPAB. Almost all come from the Chinese office (1,569 families), then from PCT international searches (172), the EPO (74), Japan (51), France (24) and the United States (6): our sector is mostly Chinese, and EPAB could not see it.

**Coverage by year.** 84% to 91% of the families of 2015–2022 have a report with letters, and about 70% of those of 2013–2014. Before 2011 coverage falls below 25%: the sector was then Japanese, American and European, and those reports rarely carry letters in PATSTAT, so we do not use the measure for these years. In the incomplete years coverage is 71% (2023), 60% (2024) and 44% (2025): the reports are still arriving, so a family without a report is a missing value, not a family without X or Y.

**The measure separates the families.** Among the families with a report, about two in three have an X or Y: 62% to 80% in 2015–2022, 57% to 65% in 2023–2025. If almost all or almost none had one, the measure would not tell families apart.

**For Step 4.**
1. Two columns per family: whether a report exists, and whether the examiner found an X, Y or I. A family without a report stays empty, never zero.
2. Families are compared only within the same priority year, and with the office of the report as a control: 92% of the reports are Chinese, so in practice the measure is the opinion of the Chinese examiners, and offices do not give the letters in the same way.
3. Before using it, we check that it works: on the families of 2015–2017, which have five complete years of citations, the families without X or Y should be cited more later. It is the same test as for the text novelty of notebook 21.